# 02 – Face embeddings

A pretrained deep network maps each face to a vector of 512 numbers, an **embedding**.
The network was trained so that photos of the same person end up close together
(high cosine similarity) and different people end up far apart.

We compare two pretrained models: **ArcFace** and **Facenet512**.

Computing embeddings for all ~3,000 images takes about 15 minutes on a CPU the first time.
They are then cached in `data/embeddings_<model>.npy`. You can also compute them from the
terminal with `python -m src.embed`.

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import src...` works from the notebooks folder

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

from src.config import EMBEDDING_MODELS, SEED
from src.data import load_lfw, split
from src.embed import get_embeddings
from src.metrics import all_pairs

sns.set_theme(style="whitegrid")
images, y, names = load_lfw()
train_idx, test_idx = split(y)

In [ ]:
E = {m: get_embeddings(m) for m in EMBEDDING_MODELS}
for m, Em in E.items():
    print(f"{m:11s} shape={Em.shape}  norms={np.linalg.norm(Em, axis=1).mean():.2f} (L2-normalised)")

## 1. Same person vs. different people

For every pair of test images we compute the cosine similarity. A good embedding gives two
clearly separated distributions. The overlap between them is where recognition errors happen.

In [ ]:
fig, axes = plt.subplots(1, len(E), figsize=(13, 4), sharey=True)
for ax, (m, Em) in zip(axes, E.items()):
    scores, same = all_pairs(Em[test_idx], y[test_idx])
    sns.histplot(x=scores, hue=np.where(same, "same person", "different people"),
                 stat="density", common_norm=False, bins=80, element="step", ax=ax)
    ax.set_title(m)
    ax.set_xlabel("Cosine similarity")
plt.tight_layout()
plt.show()

## 2. t-SNE: do the embeddings cluster by person?

t-SNE squeezes the 512 dimensions down to 2 for plotting. Each colour is one of the 10 people
with the most images.

In [ ]:
top = pd.Series(y).value_counts().index[:10]
mask = np.isin(y, top)

fig, axes = plt.subplots(1, len(E), figsize=(15, 6))
for ax, (m, Em) in zip(axes, E.items()):
    xy = TSNE(n_components=2, metric="cosine", init="pca", random_state=SEED).fit_transform(Em[mask])
    sns.scatterplot(x=xy[:, 0], y=xy[:, 1], hue=names[y[mask]], s=15, ax=ax, legend=(ax is axes[-1]))
    ax.set_title(f"t-SNE of {m} embeddings")
    ax.set_xticks([]); ax.set_yticks([])
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Observations (fill in after running)

- How well separated are the same/different distributions for each model?
- Which model gives tighter clusters in t-SNE? Any people that overlap?
- Which model do you expect to win in notebook 03, and why?